In [1]:
"""
Implementation based on: https://medium.com/we-talk-data/how-can-i-use-knn-and-random-forest-models-in-pytorch-6083f5ef370a
"""

import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

train = pd.read_csv("../data/final/dataset_full.csv")
train.head()

,Pricing Date,Issuer Name,Offer Size (M),Offer Price,Offer To 1st Close,Initial Pub Offer (Shares Offered),Industry Sector,Market Cap at Offer (M),Instit Owner (% Shares Out),Primary Exchange,...,has_bulge_bracket,vix,nasdaq,fed_funds,treasury_10y,cpi,unemployment,gdp,ipo_volume,market_return_1m
0,2000-01-24,Neoforma Inc,91.00,13.0,302.884613,7000000.0,Technology,732.744,0.014765,,...,1,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
1,2000-01-24,Townsquare Media 2010 Inc,136.00,8.5,0.000000,16000000.0,Communications,272.983,NaN,,...,0,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
2,2000-01-25,Healthgate Data Corp,41.25,11.0,6.818182,3750000.0,Communications,180.900,NaN,,...,0,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
3,2000-01-25,T/R Systems Inc,30.00,10.0,59.380001,3000000.0,Technology,115.002,NaN,,...,1,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN
4,2000-01-26,John Hancock Financial Services Inc,1734.00,17.0,3.676471,102000000.0,Financial,5638.900,0.072897,,...,1,24.95,3940.35,NaN,6.68,NaN,NaN,NaN,NaN,NaN


In [7]:
class DataPrepPipeline:
  def __init__(self):
    self.features = ['offer_size_to_mktcap', 'vix']
  def fit(self, X):
    return self
  def transform(self, X):
    eng_features = torch.from_numpy(X[self.features].values).float()
    return eng_features

In [8]:
X_df = train.drop(columns=[
    'underpriced',
    'Pricing Date',
    'Issuer Name',
    'ticker',
    'Primary Exchange'
])
y_df = train['underpriced']
X_df = pd.get_dummies(X_df, columns=['Industry Sector', 'lead_bookrunner'])
#y_df = pd.get_dummies(y_df, columns=['underpriced'])

# 80/20 split from lecture notes 9
train_ix = X_df.sample(frac=0.8, random_state=42).index
test_ix = X_df.drop(train_ix).index

X_train_df = X_df.loc[train_ix]
y_train_df = y_df.loc[train_ix]

X_test_df = X_df.loc[test_ix]
y_test_df = y_df.loc[test_ix]

pipeline = DataPrepPipeline()
pipeline.fit(X_train_df)

X_train = pipeline.transform(X_train_df)
y_train = torch.from_numpy(y_train_df.values).float()

/var/folders/ff/wt9yz2fx7dd4m0mzttpvx6300000gn/T/ipykernel_73995/1767540978.py:26: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  y_train = torch.from_numpy(y_train_df.values).float()


In [ ]:
class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2, num_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def fit(self, X, y):
        if self.num_features is None:
            self.num_features = X.shape[1]
        self.tree = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_classes = len(torch.unique(y))

        # stop conditions for recursion
        if depth >= self.max_depth or n_samples < self.min_samples_split or n_classes == 1:
            leaf_value = self._most_common_label(y) # finds most frequent class
            return {"leaf": True, "value": leaf_value}

        # select random feature subset
        feat_idxs = torch.randperm(n_features)[:self.num_features] 

        # find the best feature and threshold to split on
        best_feature, best_thresh = self._best_split(X, y, feat_idxs)

        if best_feature is None:
            return {"leaf": True, "value": self._most_common_label(y)}

        left_idxs = X[:, best_feature] <= best_thresh
        right_idxs = X[:, best_feature] > best_thresh

        left = self._grow_tree(X[left_idxs], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs], y[right_idxs], depth + 1)

        # return node
        return {
            "leaf": False,
            "feature": best_feature,
            "threshold": best_thresh,
            "left": left,
            "right": right
        }

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = None, None

        # loop thru every possible split
        for feature in feat_idxs:
            thresholds = torch.unique(X[:, feature])
            for thresh in thresholds:
                gain = self._information_gain(y, X[:, feature], thresh)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feature
                    split_thresh = thresh

        return split_idx, split_thresh

    # tests gini impurity improvement from a given split
    def _information_gain(self, y, feature_col, threshold):
        parent_loss = self._gini(y) # gini impurity before splitting (baseline)

        # boolean split masks: for each value in a feature column, check against threshold
        left_idxs = feature_col <= threshold
        right_idxs = feature_col > threshold

        if left_idxs.sum() == 0 or right_idxs.sum() == 0: # discard splits if all data goes to one side
            return 0

        n = len(y) # total samples
        n_l, n_r = left_idxs.sum(), right_idxs.sum() # number going left/right in split

        # calculate gini impurities for each split
        impurity_l = self._gini(y[left_idxs])
        impurity_r = self._gini(y[right_idxs])

        child_loss = (n_l / n) * impurity_l + (n_r / n) * impurity_r # average impurity after split
        return parent_loss - child_loss

    def _gini(self, y):
        classes = torch.unique(y)
        impurity = 1.0
        for c in classes:
            p = torch.sum(y == c).float() / len(y)
            impurity -= p ** 2
        return impurity

    def predict(self, X):
        